***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第二册：射电干涉数据处理实践](9_0_introduction.ipynb)
    * 上一节：[9.38 BIMA NGC 4826 测量集校准复盘](9_38_bima_measurement_set_calibration_replay.ipynb)
    * 下一节：[综合实践问题集（100 分）](9_problem_set.ipynb)

***


## 9.39 VLA 3C391 公开归档绝对校准实验

本节将 9.35.6 的可选纵向训练落实为一个经过真实运行验证的案例。输入为 NRAO/CASA 3C391 连续谱教程提供的 4.6 GHz 测量集（Measurement Set）：VLA 项目 `TDEM0001`，D 构型，观测日期为 2010-04-24，包含七个 3C391 镶嵌目标场。3C286 提供 Perley--Butler 2017 绝对通量模型，J1822-0938 提供随时间变化的复增益，所得标度随后转移到目标场。

本节与 9.38 的责任边界不同。9.38 使用可随仓库分发的小型 BIMA 测试测量集，训练真实数据列、数据标记、$uv$ 覆盖和相对增益的检查，但不能建立以央斯基为单位的绝对通量标度；本节具有独立归档身份和可核查的外部通量模型，因而能够训练完整的绝对标度链。由于原始压缩包约为 3.1 GB、展开后约为 5.6 GB，且 CASA 不属于基础 Python 环境，默认教材路径只审计数据清单、物理模型和参考指标，不自动下载数据或运行大型软件。


### 9.39.1 先修知识、学习目标与交付边界

先修内容包括 9.23 的归档数据适用性判断、9.27 的再处理分支选择、9.30--9.31 的运行记录与产品差异审计，以及 9.32 的发布包契约。完成本实验后，读者应能：核对归档身份、派生历史和再分发边界；区分初始权重、校准后逆方差权重与数据标记；解释延迟、带通、时间增益和绝对通量标度的求解次序；验证通量模型和校准转移；依据波束、峰值、残差与掩膜敏感性判断成像结果；提交机器可读运行记录、质量控制表和限制声明。

实验包位于 `archive_labs/vla_3c391/`。默认审计只执行本 Notebook 中的轻量代码，核对数据身份、通量模型和实测参考指标；完整纵向运行则需另行下载归档测量集，在 CASA 环境中完成校准、成像敏感性实验和 9.39.7 的 100 分课程任务。前者验证教材包及证据契约，后者才验证学生自己的处理链，二者不能相互替代。

仓库只分发 GPLv2 脚本、数据清单和项目实测参考指标，不分发 NRAO 测量集，也不将“公开下载”解释为“可按 GPLv2 再许可”。这一边界本身就是数据管理训练的一部分。


In [ ]:
import importlib.util
from pathlib import Path

import yaml

lab_dir = Path('archive_labs/vla_3c391')
if not lab_dir.exists():
    lab_dir = Path('9_Practical') / lab_dir
manifest = yaml.safe_load((lab_dir / 'manifests/data_manifest.yaml').read_text())
reference = yaml.safe_load((lab_dir / 'manifests/reference_metrics.yaml').read_text())
spec = importlib.util.spec_from_file_location('vla_3c391_audit', lab_dir / 'audit_results.py')
audit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit)

print(manifest['sample_id'])
print(manifest['source']['archive_execution_block'])
print(f"Archive: {manifest['archive']['bytes'] / 1024**3:.2f} GiB")
print(manifest['source']['redistribution_status'])


### 9.39.2 数据状态与第一处理步骤

教程测量集有 845,379 行、26 根天线、64 个 2 MHz 通道，覆盖 4.536--4.662 GHz，积分时间为 10 s，保留 RR、RL、LR、LL。它从原观测的 4.6 和 7.5 GHz 双谱窗中只保留前者，并从 1 s 平均到 10 s；因此它是具有明确派生历史的教学起点，而不是未经处理的完整归档交付物。七个目标场之外，还包括 3C286、J1822-0938 和 3C84。

初始 `FLAG` 全为假只表示该测量集中尚未记录数据标记，并不表示所有数据均合格。初始 `CORRECTED_DATA` 等于 `DATA`，说明校准尚未转移；`WEIGHT_SPECTRUM` 虽然存在但尚未初始化，起始 `WEIGHT` 也不能直接解释为可靠的 Jy$^{-2}$ 逆方差。因此，实验先进行人工数据标记和校准，在目标拆分后再运行 `statwt`。若改变这一因果顺序，成像器得到的数值权重虽然形式合法，其物理含义却不成立。


### 9.39.3 绝对通量模型是外部证据

Perley-Butler 2017 对 3C286 使用以 GHz 为频率单位的多项式

$$\log_{10} S_\nu = \sum_{k=0}^{3} a_k[\log_{10}(\nu/\mathrm{GHz})]^k,$$

其中 $S_\nu$ 以 Jy 为单位，本实验固定 $[a_0,a_1,a_2,a_3]=[1.2481,-0.4507,-0.1798,0.0357]$。这个模型把无量纲相关系数链锚定到物理通量密度；J1822-0938 的 `fluxscale` 结果再把该标度传到时间增益。`fluxscale` 给出的统计散布不是总误差，3C286 模型系统误差、带通误差、时间插值和目标方向差异仍须单列。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

checkpoints = reference['flux_model']['checkpoints']
frequency_ghz = np.array([point['frequency_ghz'] for point in checkpoints])
expected_jy = np.array([point['flux_jy'] for point in checkpoints])
model_jy = np.array([audit.perley_butler_2017_3c286(value * 1e9) for value in frequency_ghz])
np.testing.assert_allclose(model_jy, expected_jy, atol=1e-6)

frequency_grid = np.linspace(4.53, 4.67, 200)
flux_grid = [audit.perley_butler_2017_3c286(value * 1e9) for value in frequency_grid]
plt.figure(figsize=(7, 4))
plt.plot(frequency_grid, flux_grid)
plt.scatter(frequency_ghz, expected_jy, color='tab:red', zorder=3)
plt.xlabel('Frequency [GHz]')
plt.ylabel('3C286 flux density [Jy]')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 9.39.4 校准链与规范选择

脚本首先对扫描 1、天线 ea13/ea15 和每个扫描开头的 10 s 数据作标记，并采用教程给定的天线位置偏移。随后对 3C286 运行 `setjy`，用窄通道区间求初始相位，在有效通道上求 K 型延迟和复带通，再重新求解 3C286 与 J1822-0938 的幅相增益。参考天线 ea21 固定了不可观测的整体相位规范。

对基线 $pq$，方向无关标量近似为

$$V_{pq}^{\mathrm{obs}}(\nu,t)=g_p(t)g_q^*(t)b_p(\nu)b_q^*(\nu)e^{-2\pi i\nu(\tau_p-\tau_q)}V_{pq}^{\mathrm{model}}+n_{pq}.$$

分步求解并不意味着这些项严格独立，而是利用不同的时间和频率平滑尺度约束自由度。较强的残余延迟会表现为带通相位斜率，错误的源模型会进入增益幅度，归一化选择还会在不同校准表之间转移尺度。因此，每张校准表都应检查有效解数、数据标记比例、幅相连续性和参考天线，而不能只检查最终图像。标度转移后只拆分七个目标场的 RR/LL；在忽略 Stokes $V$ 且圆馈源泄漏足够小时，Stokes $I\simeq(RR+LL)/2$。本实验不使用 RL/LR 开展偏振测量。


In [ ]:
calibration = reference['calibration']
print('3C286 setjy [Jy]:', calibration['setjy_3c286_flux_jy']['value'])
print('J1822-0938 [Jy]:', calibration['gain_calibrator_flux_jy']['value'])
print('Fluxscale solutions:', calibration['valid_fluxscale_solutions'])
print('Science flag fraction:', calibration['science_flag_fraction']['value'])
print('Fully flagged antennas:', ', '.join(calibration['fully_flagged_antennas']))
print('Delay range [ns]:', calibration['delay_solution_ns']['minimum']['value'],
      calibration['delay_solution_ns']['maximum']['value'])


### 9.39.5 参考数值与回归诊断边界

CASA 6.7.0.31 验证运行得到：3C286 在 4.536 GHz 为 7.66855 Jy；J1822-0938 在 4.599 GHz 为 $2.29601\pm0.00692$ Jy，共 46 个有效标度解；有效延迟约为 $-3.85$ 到 $+4.56$ ns。目标拆分后的总数据标记比例约为 34.55%，ea05、ea13、ea15 的数据全部被标记，七个目标场的数据标记比例分别约为 33.5%--35.2%。

这些数值可用于发现数据版本、选择表达式、参考天线或任务行为的意外变化，但“落在容差内”并不能自动证明校准可信。学生还应绘制校准解随天线、时间和通道的变化，检查断点、相位绕转、孤立天线和校准源残差；任何新增的数据标记都必须说明证据及其对求解约束的影响。


### 9.39.6 成像基线中的受控失败模式

参考成像使用 $480\times480$ 个像元、2.5 arcsec 像元尺度、镶嵌网格化器、Briggs 加权（`robust=0.5`）和多尺度参数 `[0,5,15,45]`。验证运行的恢复波束约为 $17.0\times14.9$ arcsec，位置角约为 $21.0^\circ$；脏图峰值约为 0.1229 Jy beam$^{-1}$，恢复图峰值约为 0.1301 Jy beam$^{-1}$，全图残差均方根约为 0.570 mJy beam$^{-1}$，残差极值约为 $-2.27$ 到 $+2.24$ mJy beam$^{-1}$。固定的 130 像素半径圆形掩膜在次循环中产生 11 次大尺度负分量或发散警告，模型总和约为 3.788 Jy，其中负分量占模型绝对通量的约 25.6%。

受控实验保持可见度、网格、权重、阈值和恢复波束不变。100 像素半径的紧凑圆形掩膜将负分量比例降至 12.9%，但残差均方根升至 0.840 mJy beam$^{-1}$，掩膜边界出现明显弧状残差；保守的 `auto-multithresh` 将负分量比例降至 0.7%，却把模型总和推高到 8.070 Jy，产生 17 次发散警告，残差均方根仍为 0.645 mJy beam$^{-1}$，且 45 像素尺度曾因无法放入局部掩膜而被忽略。保持大圆形掩膜、只删除 45 像素尺度可以消除发散警告，但残差均方根为 0.737 mJy beam$^{-1}$，一个波束尺度上的相关系数升至约 0.85/0.90，说明扩展结构主要留在残差中。

因此，四组方案均不能支持更强的科学产品。较少的负 CLEAN 分量、较低的均方根或没有发散警告都只是单项证据，不能单独决定方案优劣；掩膜还会改变有效模型空间，因而不能把不同方案的模型通量视为相互独立的同一估计量。固定大圆形掩膜只保留为可复现回归基线，发表级结论的状态仍为 `false`。若要进一步测量目标通量，必须检查可见度与点扩散函数对大尺度结构的约束，独立审查源支撑区，并处理主波束、相关噪声和镶嵌观测的空间响应。


In [ ]:
imaging = reference['imaging']
beam = imaging['restoring_beam']
print('Beam [arcsec]:', beam['major_arcsec']['value'], 'x', beam['minor_arcsec']['value'])
print('Restored peak [Jy/beam]:', imaging['restored_peak_jy_per_beam']['value'])
print('Residual RMS [mJy/beam]:', 1e3 * imaging['residual_rms_jy_per_beam']['value'])

sensitivity = reference['imaging_sensitivity']
for name in ('fixed_broad', 'fixed_tight', 'auto_conservative',
             'fixed_broad_restricted_scales'):
    result = sensitivity[name]
    print(name, 'model [Jy]:', result['model_sum_jy'],
          'residual [mJy/beam]:', result['residual_rms_mjy_per_beam'])
print('Sensitivity conclusion:', sensitivity['conclusion']['status'])
print('Publication claim:', sensitivity['conclusion']['publication_claim_supported'])


### 9.39.7 课程纵向训练（100 分）

1. **归档与许可（10 分）**：核对项目、执行块、观测日期、阵列构型、教程派生历史、URL、字节数和 SHA-256；写明为何本仓库不再分发测量集。
2. **初始测量集审计（10 分）**：报告主表行数、场、天线、谱窗、通道、相关积、积分时间和关键列；说明初始 `FLAG`、`CORRECTED_DATA` 与权重分别允许或不允许得出哪些结论。
3. **数据标记证据（10 分）**：按扫描、天线、场、时间和通道比较标记前后的统计量，解释 ea05/ea13/ea15 的处理如何改变求解约束。
4. **绝对标度（15 分）**：复现 3C286 多项式，核对 `setjy` 和 J1822-0938 `fluxscale`，区分形式统计误差与通量模型、标度转移和方向相关系统误差。
5. **校准表质量控制（20 分）**：检查 G0、K0、B0、G1 与 `fluxscale` 表的有效解、幅相连续性、频率结构、参考天线和校准源残差；至少给出一个失败判据。
6. **转移与权重（10 分）**：说明目标 `gainfield` 和 `interpolation` 的选择，拆分 RR/LL 后运行 `statwt`；验证权重、`FLAG` 与数据形状一致。
7. **成像与失败模式（15 分）**：运行 `run_imaging_sensitivity.py` 或等价的受控实验，比较掩膜面积、模型正负通量、残差均方根与极值、空间相关、有效尺度和停止日志；说明为何本节四个实测方案均不应升级为发表产品，并提出下一项能够区分成因的检验。
8. **复现报告（10 分）**：提交软件版本、CASA 数据路径配置、命令、参数、运行日志、JSON 审计报告、产品校验和与限制声明。

评分依据是证据链，而不是最终图像的外观。校准或成像未通过时，正确停止、定位失败并降低结论等级可以获得相应判断分；隐去错误或用参考数值替代自己的输出不能得分。


### 9.39.8 运行与审计

`download_data.py` 支持断点续传、长度与 SHA-256 校验，并在解包前拒绝越界路径、链接和设备文件。`run_casa_pipeline.py` 将输入复制到空工作目录，设置校准和成像两个停止点，并保存 CASA 版本、参数、数据标记、`fluxscale`、延迟、`statwt`、成像统计和恢复波束。`run_imaging_sensitivity.py` 复用校准后的目标测量集，在独立空目录中运行三种掩膜方案和一项尺度诊断，输出 `imaging_sensitivity.json`；它不会重复约 6 GB 的校准链。`audit_results.py` 使用带容差的参考指标生成通过或失败报告；`--require-imaging` 可要求审计完整处理链。详细命令见实验包 [README](archive_labs/vla_3c391/README.md)。

参考指标只用于验证同一输入和同一基线流程是否一致。更换 CASA 版本、数据标记、通量模型、参考天线、掩膜或成像参数后，应保留新的运行身份并解释差异，不应为了“通过测试”而把科学上合理的变化强行调回旧值。反过来，任何新流程也必须重新建立可核查的数值基线。


### 9.39.9 本节结论

3C391 案例补齐了从公开归档身份、外部绝对通量模型、延迟、带通和时间增益到镶嵌成像与最终质量控制的纵向训练。它为教材建立了一套经过真实数据验证的课程实验入口，同时没有把大型第三方数据、CASA 或单一参考图像变成基础学习的前置条件。

更重要的是，本实验没有把“处理流水线运行结束”等同于“科学结论完成”。绝对标度必须追溯到外部模型，权重必须在校准后的数据语义下重新估计，成像必须报告掩膜与残差的失败模式。四组受控实验均未产生明确更优的图像，这不是需要隐去的失败，而是可评分的停止决定：不同掩膜和尺度在负模型、通量、发散与相关残差之间交换风险，现有证据不足以支持发表级测量。只有区分参考指标与科学验收，并据此降低结论等级，才算完成一次可审查的射电干涉数据处理复盘。

***
